In [ ]:

import os, sys, glob, shutil, time, json
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.0f}с] {m}", flush=True)
# Каталог датасета опознаём по файлу, которого нет больше нигде: модули с тем же именем
# лежат и в выводе прошлого ядра, и glob находил именно его.
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(base+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.makedirs("/kaggle/working/models",exist_ok=True)
shutil.copy(base+"/anti_words.json","/kaggle/working/models/anti_words.json")
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.features import extract_model_features
from src.hybrid import product_disjoint_pair_masks
from src.metrics import macro_pr_auc
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

items=pd.read_parquet(base+"/items_human.parquet")
matches=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
evalp=pd.read_parquet(base+"/eval_pairs.parquet")
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_new.npy",recursive=True)[0])
Xnew=np.load(prev+"/features_new.npy"); Enew=np.load(prev+"/features_eval.npy")
log(f"новые признаки: обучение {Xnew.shape}, линейка {Enew.shape}")

# Старые 168 признаков для пар линейки: их не было, поэтому главное сравнение не сходилось.
t=time.perf_counter()
Eold=extract_model_features(evalp[["id1","id2"]], items)
log(f"168 признаков для линейки: {Eold.shape} за {time.perf_counter()-t:.0f}с")
np.save("/kaggle/working/eval_features_old.npy",Eold)
Xold=np.load(base+"/human_features_v13.npy")

y=matches["target"].to_numpy(np.int8)
cat_of=dict(zip(items["id"],items["category"].astype(str)))
cp=matches["id1"].map(cat_of).astype(str).to_numpy()
tm,vm=product_disjoint_pair_masks(matches["id1"].to_numpy(),matches["id2"].to_numpy(),0,3)
tr,va=np.flatnonzero(tm),np.flatnonzero(vm)
ye=evalp["target"].to_numpy(np.int8); ce=evalp["category"].astype(str).to_numpy()
def macro_eval(p): return float(np.mean([average_precision_score(ye[ce==k],p[ce==k])
    for k in np.unique(ce) if len(np.unique(ye[ce==k]))>1]))
for tag,Xt,Xe in (("168 старых",Xold,Eold),("97 новых",Xnew,Enew),
                  ("168 + 97",np.hstack([Xold,Xnew]),np.hstack([Eold,Enew]))):
    clf=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.08,random_state=0)
    clf.fit(Xt[tr],y[tr])
    p=clf.predict_proba(Xt[va])[:,1]; pe=clf.predict_proba(Xe)[:,1]
    np.save(f"/kaggle/working/eval_pred_{tag.replace(' ','_').replace('+','p')}.npy",pe)
    log(f"{tag:<12} holdout {macro_pr_auc(y[va],p,cp[va])[0]:.6f}   линейка {macro_eval(pe):.6f}")
log("готово")
